In [0]:
# load storage config
%run ../includes/storage_config.ipynb

In [0]:
# load common functions
from utils.common_functions import clean_column_names

In [0]:
# Manually add schema to ensure fields are parsed correctly
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType

source_schema = StructType(fields = [
    StructField("Service:RDT-ID", StringType(), False), #service id cannot be null
    StructField("Service:Date", StringType(), True),
    StructField("Service:Type", StringType(), True),
    StructField("Service:Company", StringType(), True),
    StructField("Service:Train number", StringType(), True),
    StructField("Service:Completely cancelled", BooleanType(), True),
    StructField("Service:Partly cancelled", BooleanType(), True),
    StructField("Service: Maximum delay", IntegerType(), True),
    StructField("Stop:RDT-ID", StringType(), True),
    StructField("Stop:Station code", StringType(), True),
    StructField("Stop:Station name", StringType(), True),
    StructField("Stop:Arrival time", StringType(), True),
    StructField("Stop:Arrival delay", IntegerType(), True),
    StructField("Stop:Arrival cancelled", BooleanType(), True),
    StructField("Stop:Departure time", StringType(), True),
    StructField("Stop:Departure delay", IntegerType(), True),
    StructField("Stop:Departure cancelled", BooleanType(), True),
    StructField("Stop:Platform change", BooleanType(), True),
    StructField("Stop:Planned platform", StringType(), True),
    StructField("Stop:Actual platform", StringType(), True)
    ])

### Step 1 - fetch data in csv format

In [0]:
# read the csv file
source_df = spark.read.schema(source_schema).csv(source_file_path, header=True)

In [0]:
# quick glance at the dataframe
source_df.limit(5)

### Step 2 - rename the columns for consistency

In [0]:
renamed_df = clean_column_names(source_df)


### Step 3 - Add ingestion time stamp

In [0]:
from pyspark.sql.functions import lit, current_timestamp
final_df = renamed_df.withColumn("file_source", lit("csv")) \
    .withColumn("ingestion_date", current_timestamp()) 

### Step 4 - write data to delta lake, create managed table

In [0]:
raw_table_location = f'{schema}.{raw_table_name}'
final_df.write.mode("overwrite").format("delta").saveAsTable(raw_table_location)

In [0]:
sql = f'''
    SELECT * FROM {raw_table_location}
    LIMIT 5;
'''
df = spark.sql(sql)
display(df.limit(5))

In [0]:
dbutils.notebook.exit("Success")